# Basic RAG Implementation with Groq

This notebook builds a complete **Retrieval-Augmented Generation (RAG)** application using maintained LangChain integrations:

- **Groq** generates the final answer.
- **Hugging Face** creates embeddings locally without paid embedding API calls.
- **Chroma** stores and searches document vectors.
- **LCEL** connects retrieval, prompting, generation, and parsing.

## Complete flow

```text
Text file → Document → Chunks → Local embeddings → Chroma
                                                   ↓
Question → Retriever → Relevant chunks → Prompt → Groq → Answer + sources
```

> Select the **Python 3.13.7 (`.venv`)** kernel and run cells from top to bottom.

## 1. Configure the application

### What?

Configuration includes model names, chunk settings, retrieval count, timeouts, and output limits.

### Why?

Enterprise applications keep tunable values outside the processing logic. A typed, immutable settings object validates configuration once at startup.

Add these values to `.env` when you want to override the defaults:

```text
GROQ_API_KEY=your_groq_api_key
GROQ_MODEL=qwen/qwen3.8-27b
GROQ_MAX_TOKENS=256
EMBEDDING_MODEL=sentence-transformers/all-MiniLM-L6-v2
RAG_CHUNK_SIZE=800
RAG_CHUNK_OVERLAP=120
RAG_TOP_K=3
GROQ_TIMEOUT_SECONDS=30
```

Never commit your real `.env` file.

In [1]:
import logging
import os
from dataclasses import dataclass
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(), override=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    force=True,
)
logger = logging.getLogger("basic_rag")


def positive_int_from_env(name: str, default: int) -> int:
    raw_value = os.getenv(name, str(default))
    try:
        value = int(raw_value)
    except ValueError as error:
        raise ValueError(f"{name} must be an integer, received {raw_value!r}.") from error
    if value <= 0:
        raise ValueError(f"{name} must be greater than zero.")
    return value


@dataclass(frozen=True, slots=True)
class Settings:
    groq_model: str
    embedding_model: str
    max_output_tokens: int
    chunk_size: int
    chunk_overlap: int
    top_k: int
    groq_timeout_seconds: int

    @classmethod
    def from_environment(cls) -> "Settings":
        if not os.getenv("GROQ_API_KEY"):
            raise RuntimeError("GROQ_API_KEY is missing from the .env file.")

        settings = cls(
            groq_model=os.getenv("GROQ_MODEL", "qwen/qwen3.8-27b"),
            embedding_model=os.getenv(
                "EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2"
            ),
            max_output_tokens=positive_int_from_env("GROQ_MAX_TOKENS", 256),
            chunk_size=positive_int_from_env("RAG_CHUNK_SIZE", 800),
            chunk_overlap=positive_int_from_env("RAG_CHUNK_OVERLAP", 120),
            top_k=positive_int_from_env("RAG_TOP_K", 3),
            groq_timeout_seconds=positive_int_from_env("GROQ_TIMEOUT_SECONDS", 30),
        )
        if settings.chunk_overlap >= settings.chunk_size:
            raise ValueError("RAG_CHUNK_OVERLAP must be smaller than RAG_CHUNK_SIZE.")
        return settings


settings = Settings.from_environment()
logger.info("Configuration loaded for Groq model %s", settings.groq_model)

2026-10-01 12:53:11,446 | INFO | basic_rag | Configuration loaded for Groq model qwen/qwen3.8-27b


## 2. Load the source document

### What is a LangChain `Document`?

A `Document` has two important parts:

| Field | Meaning |
| --- | --- |
| `page_content` | Text used by the RAG pipeline |
| `metadata` | Source information retained with the text |

### When and why?

Convert all input sources into `Document` objects so downstream components receive one consistent format. Metadata is essential for debugging, access filtering, and showing sources to users.

This example reads a trusted local text file directly instead of using the deprecated `langchain-community` loader.

In [2]:
from langchain_core.documents import Document


def load_text_document(path: str | Path) -> list[Document]:
    file_path = Path(path)
    if not file_path.is_file():
        raise FileNotFoundError(f"Source file was not found: {file_path.resolve()}")

    content = file_path.read_text(encoding="utf-8").strip()
    if not content:
        raise ValueError(f"Source file is empty: {file_path}")

    logger.info("Loaded %d characters from %s", len(content), file_path)
    return [Document(page_content=content, metadata={"source": str(file_path)})]


source_documents = load_text_document("data/state_of_the_union.txt")
print("Documents loaded:", len(source_documents))
print("Metadata:", source_documents[0].metadata)
print("Preview:\n", source_documents[0].page_content[:400])

2026-10-01 12:53:20,773 | INFO | basic_rag | Loaded 38539 characters from data\state_of_the_union.txt


Documents loaded: 1
Metadata: {'source': 'data\\state_of_the_union.txt'}
Preview:
 Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  

Last year COVID-19 kept us apart. This year we are finally together again. 

Tonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. 

With a duty to one another to the American people to the Con


## 3. Split the document into chunks

### What is chunking?

Chunking divides long text into smaller searchable units.

### Why use overlap?

A sentence or idea can cross a chunk boundary. `chunk_overlap` repeats a small amount of neighboring text so meaning is less likely to be lost.

### Why use `RecursiveCharacterTextSplitter`?

It tries paragraph, line, word, and character boundaries in order. This usually preserves natural text structure better than splitting with only one separator.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=settings.chunk_size,
    chunk_overlap=settings.chunk_overlap,
    length_function=len,
    add_start_index=True,
)
chunks = text_splitter.split_documents(source_documents)

for chunk_number, chunk in enumerate(chunks):
    chunk.metadata["chunk_number"] = chunk_number

if not chunks:
    raise RuntimeError("Text splitting produced no chunks.")

logger.info("Created %d chunks", len(chunks))
print("Total chunks:", len(chunks))
print("First chunk metadata:", chunks[0].metadata)
print("First chunk preview:\n", chunks[0].page_content[:500])

e:\AamadNaseem\MSAIEngineering\AI Engineer\AiProjects\Latest-LangChain-LLMOps\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-10-01 12:53:35,953 | INFO | basic_rag | Created 57 chunks


Total chunks: 57
First chunk metadata: {'source': 'data\\state_of_the_union.txt', 'start_index': 0, 'chunk_number': 0}
First chunk preview:
 Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  

Last year COVID-19 kept us apart. This year we are finally together again. 

Tonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. 

With a duty to one another to the American people to the Constitution. 

And with an unwavering resolve that freedom will always triumph over tyranny. 

Six day


## 4. Create local embeddings

### What is an embedding?

An embedding is a numeric vector representing text meaning. Similar meanings should produce vectors that are close together.

### When and why?

Embeddings make semantic search possible. A question can retrieve a useful passage even when it does not use exactly the same words.

### Why Hugging Face instead of Groq embeddings?

Groq provides chat inference but no embedding API. `HuggingFaceEmbeddings` runs the embedding model locally, so no paid OpenAI embedding credit is required.

> The first run downloads the public embedding model; later runs use the local cache. In production, pin and pre-package an approved model version.

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings_model = HuggingFaceEmbeddings(
    model_name=settings.embedding_model,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

sample_vector = embeddings_model.embed_query("What are voting rights?")
print("Embedding dimensions:", len(sample_vector))
print("First five values:", sample_vector[:5])

2026-10-01 12:53:45,387 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-10-01 12:53:45,435 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2Fmain%2Fmodules.json=&etag=%22952a9b81c0bfd99800fabf352f69c7ccd46c5e43%22 "HTTP/1.1 200 OK"
2026-10-01 12:53:45,696 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-10-01 12:53:45,698 | WARNING | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-10-01 12:53:45,750 | INFO | httpx | HTTP Request: HEAD htt

Embedding dimensions: 384
First five values: [-0.0149300005286932, 0.010023420676589012, -0.07121047377586365, 0.03416276350617409, -0.020610012114048004]


## 5. Build the Chroma vector store

### What is a vector store?

A vector store keeps chunks, metadata, and their embeddings. It searches for vectors closest to a question vector.

### Why Chroma?

Chroma is easy to use locally and has a maintained dedicated LangChain package. This notebook uses an in-memory collection for repeatable learning.

In production, consider persistence, stable IDs, document versions, tenant filters, backups, and a re-indexing strategy.

In [5]:
from langchain_chroma import Chroma

chunk_ids = [f"state-of-the-union-{index:04d}" for index in range(len(chunks))]
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings_model,
    ids=chunk_ids,
    collection_name="basic_rag_demo",
)
logger.info("Indexed %d chunks in Chroma", len(chunks))

2026-10-01 12:53:55,030 | INFO | basic_rag | Indexed 57 chunks in Chroma


## 6. Create and test the retriever

### What is a retriever?

A retriever accepts a question and returns relevant `Document` objects through a standard LangChain interface.

### What does `top_k` mean?

`top_k` is the number of chunks returned. Too few can miss supporting evidence; too many add noise and consume prompt tokens.

In [6]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": settings.top_k},
)

test_question = "What did the president say about Ketanji Brown Jackson?"
retrieved_documents = retriever.invoke(test_question)

if not retrieved_documents:
    raise RuntimeError("The retriever returned no documents.")

print("Retrieved chunks:", len(retrieved_documents))
for rank, document in enumerate(retrieved_documents, start=1):
    print(f"\n--- Result {rank} ---")
    print("Metadata:", document.metadata)
    print(document.page_content[:400])

Retrieved chunks: 3

--- Result 1 ---
Metadata: {'start_index': 29345, 'source': 'data\\state_of_the_union.txt', 'chunk_number': 43}
One of the most serious constitutional responsibilities a President has is nominating someone to serve on the United States Supreme Court. 

And I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence. 

A former top litigator in private practice. A former federal publ

--- Result 2 ---
Metadata: {'source': 'data\\state_of_the_union.txt', 'chunk_number': 51, 'start_index': 34885}
Committed to military families like Danielle Robinson from Ohio. 

The widow of Sergeant First Class Heath Robinson.  

He was born a soldier. Army National Guard. Combat medic in Kosovo and Iraq. 

Stationed near Baghdad, just yards from burn pits the size of football fields. 

Heath’s widow Danielle is here with us tonight. They loved going to Ohio 

## 7. Configure Groq and the grounded prompt

The embedding model searches documents. The Groq chat model performs a different job: it reads retrieved context and writes the answer.

A grounded prompt tells the model to:

- use only the supplied context;
- say when the answer is unavailable;
- avoid inventing unsupported facts;
- mention supporting source numbers.

A low temperature is preferred for factual RAG responses.

In [7]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

chat_model = ChatGroq(
    model=settings.groq_model,
    temperature=0.1,
    reasoning_effort="none",
    max_tokens=settings.max_output_tokens,
    timeout=settings.groq_timeout_seconds,
    max_retries=2,
)

rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a factual question-answering assistant. "
            "Use only the supplied context. If the answer is not present, "
            "say that you do not know from the provided context. "
            "Do not invent facts. Cite supporting sections as [Source N].",
        ),
        ("human", "Question: {question}\n\nContext:\n{context}"),
    ]
)

## 8. Connect the complete RAG pipeline with LCEL

### What is LCEL?

**LangChain Expression Language** connects runnable components with the `|` operator.

### What happens in this chain?

1. `RunnableParallel` retrieves documents while preserving the question.
2. `RunnablePassthrough.assign()` adds formatted context.
3. Another assignment generates the answer with the prompt, Groq, and parser.
4. The result keeps both the answer and retrieved documents for traceability.

```text
question ─┬─→ retriever → documents → formatted context ─┐
          └─→ original question ────────────────────────┼→ prompt → Groq → answer
```

> **Privacy:** Invoking the final chain sends retrieved excerpts from the local document to Groq. Run it only when you are authorized to share that data with the configured provider.

In [8]:
from operator import itemgetter

from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough


def format_documents(documents: list[Document]) -> str:
    if not documents:
        return "No relevant context was retrieved."

    sections = []
    for source_number, document in enumerate(documents, start=1):
        source = document.metadata.get("source", "unknown")
        chunk_number = document.metadata.get("chunk_number", "unknown")
        sections.append(
            f"[Source {source_number}: {source}, chunk {chunk_number}]\n"
            f"{document.page_content}"
        )
    return "\n\n".join(sections)


answer_chain = (
    {
        "question": itemgetter("question"),
        "context": itemgetter("context"),
    }
    | rag_prompt
    | chat_model
    | StrOutputParser()
)

rag_chain = (
    RunnableParallel(
        question=RunnablePassthrough(),
        documents=retriever,
    )
    | RunnablePassthrough.assign(
        context=RunnableLambda(lambda data: format_documents(data["documents"]))
    )
    | RunnablePassthrough.assign(answer=answer_chain)
)

In [9]:
def ask_rag(question: str) -> dict:
    clean_question = question.strip()
    if not clean_question:
        raise ValueError("Question cannot be empty.")

    logger.info("Running RAG query with top_k=%d", settings.top_k)
    return rag_chain.invoke(clean_question)


result = ask_rag("What did the president say about Ketanji Brown Jackson?")

print("Answer:\n")
print(result["answer"])
print("\nSources used:")
for index, document in enumerate(result["documents"], start=1):
    print(f"{index}. {document.metadata}")

2026-10-01 12:55:02,418 | INFO | basic_rag | Running RAG query with top_k=3
2026-10-01 12:55:03,265 | INFO | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Answer:

The President stated that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson four days prior to the speech, describing her as "one of our nation’s top legal minds" who would "continue Justice Breyer’s legacy of excellence" [Source 1]. He also highlighted her background as a former top litigator in private practice and a former federal public defender, noting that she comes from a family of public school educators and police officers [Source 1]. Additionally, the President described her as a "consensus builder" who has received broad support from various groups, including the Fraternal Order of Police and former judges appointed by both Democrats and Republicans [Source 1].

Sources used:
1. {'source': 'data\\state_of_the_union.txt', 'chunk_number': 43, 'start_index': 29345}
2. {'source': 'data\\state_of_the_union.txt', 'chunk_number': 51, 'start_index': 34885}
3. {'start_index': 28635, 'source': 'data\\state_of_the_union.txt', 'chunk_number': 42}


## 9. Why return the answer and source documents?

Returning only an answer makes troubleshooting and review difficult. Keeping the retrieved documents allows you to:

- show sources to the user;
- check whether retrieval found the right evidence;
- record an audit trail;
- measure retrieval quality separately from generation quality.

## Production checklist

| Area | Current notebook | Production extension |
| --- | --- | --- |
| Secrets | `.env` | Managed secret store and key rotation |
| Configuration | Typed validation | Environment-specific configuration service |
| Logging | Structured stage logs | Centralized logs, tracing, metrics, and alerts |
| Reliability | Timeout and retries | Rate-limit handling, circuit breaker, and provider fallback |
| Data lifecycle | In-memory index | Persistent storage, versions, deletion, and re-indexing |
| Security | Trusted local source | File validation, access filters, encryption, and prompt-injection controls |
| Quality | Grounded prompt | Retrieval tests, answer evaluations, and human review |

**Main lesson:** RAG quality depends on loading, chunking, embeddings, retrieval, and generation together—not only on the chat model.